# Feature-set similarity effect analysis

This read-only analysis compares `scalar_compact` with other feature sets under matched evaluation conditions. It visualizes overall gains and whether gains are larger for smaller scorer LLMs.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from notebooks.utils.classification_results_analysis_utils import load_classification_run
from notebooks.utils.feature_set_similarity_effect_utils import (
    DEFAULT_MODEL_SIZES_BILLIONS, aggregate_feature_set_deltas, attach_model_sizes,
    matched_feature_set_deltas, plot_delta_by_model_size, plot_feature_set_comparison,
    plot_feature_set_delta_heatmap,
)

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
DEFAULT_SOURCE_RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
CLASSIFIER_ANALYSIS_SOURCE_RUN_ID = os.environ.get('CLASSIFIER_ANALYSIS_SOURCE_RUN_ID', DEFAULT_SOURCE_RUN_ID)
CLASSIFIER_ANALYSIS_RUN_IDS = os.environ.get('CLASSIFIER_ANALYSIS_RUN_IDS', CLASSIFIER_ANALYSIS_SOURCE_RUN_ID)
RUN_IDS = [run_id.strip() for run_id in CLASSIFIER_ANALYSIS_RUN_IDS.split(',') if run_id.strip()]
BASELINE_FEATURE_SET = os.environ.get('FEATURE_EFFECT_BASELINE', 'scalar_compact')
COMPARISON_FEATURE_SETS = tuple(name.strip() for name in os.environ.get('FEATURE_EFFECT_COMPARISONS', 'simple_similarity,extended,scalar_all').split(',') if name.strip())
METRIC = os.environ.get('FEATURE_EFFECT_METRIC', 'macro_f1')
OUTPUT_ROOT = Path(os.environ.get('FEATURE_EFFECT_OUTPUT_ROOT', str(DATA_ROOT / 'experiment-results' / 'classification' / 'analysis' / 'feature-set-similarity-effect')))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
frames = []
warnings = []
for run_id in RUN_IDS:
    try:
        run = load_classification_run(DATA_ROOT, run_id=run_id)
    except (OSError, ValueError) as error:
        warnings.append(f'{run_id}: {error}')
        continue
    metrics = run.tables.get('evaluation_metrics', pd.DataFrame()).copy()
    if not metrics.empty:
        metrics['source_run_id'] = run_id
        frames.append(metrics)
metrics = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if warnings:
    display(pd.DataFrame({'warning': warnings}))
if metrics.empty:
    raise ValueError('No evaluation_metrics rows were loaded from the configured run IDs.')
available_feature_sets = sorted(metrics['feature_set'].dropna().astype(str).unique())
comparison_feature_sets = tuple(name for name in COMPARISON_FEATURE_SETS if name in available_feature_sets)
missing_feature_sets = sorted(set(COMPARISON_FEATURE_SETS).difference(comparison_feature_sets))
if missing_feature_sets:
    display(pd.DataFrame({'missing_comparison_feature_set': missing_feature_sets}))
print(f'Loaded {len(metrics):,} evaluation rows for {metrics["scorer_llm"].nunique()} scorer LLMs.')


In [ ]:
deltas = matched_feature_set_deltas(
    metrics,
    baseline_feature_set=BASELINE_FEATURE_SET,
    comparison_feature_sets=comparison_feature_sets,
    metric=METRIC,
)
summary = attach_model_sizes(aggregate_feature_set_deltas(deltas))
coverage = (
    metrics.groupby(['scorer_llm', 'feature_set'], as_index=False)
    .size().rename(columns={'size': 'evaluation_row_count'})
)
deltas.to_csv(OUTPUT_ROOT / 'matched_feature_set_deltas.csv', index=False)
summary.to_csv(OUTPUT_ROOT / 'feature_set_effect_summary.csv', index=False)
coverage.to_csv(OUTPUT_ROOT / 'feature_set_coverage.csv', index=False)
display(summary)


In [ ]:
figures = {
    'feature_set_comparison.png': plot_feature_set_comparison(
        deltas, baseline_feature_set=BASELINE_FEATURE_SET, title=f'{METRIC} comparison',
    ),
    'feature_set_delta_heatmap.png': plot_feature_set_delta_heatmap(
        summary, title=f'{METRIC} gain over {BASELINE_FEATURE_SET}',
    ),
    'delta_by_model_size.png': plot_delta_by_model_size(
        summary, title=f'{METRIC} gain versus scorer model size',
    ),
}
for filename, figure in figures.items():
    display(figure)
    figure.savefig(OUTPUT_ROOT / filename, dpi=160, bbox_inches='tight')
    plt.close(figure)
unknown_sizes = sorted(set(summary.loc[~summary['size_known'], 'scorer_llm'])) if not summary.empty else []
if unknown_sizes:
    display(pd.DataFrame({'unknown_model_size': unknown_sizes}))
print(f'Wrote analysis outputs to {OUTPUT_ROOT}')
